# nnU-Net Baseline Architecture
This notebook contains the implementation of the nnU-Net medical standard baseline.
It uses Instance Normalization and LeakyReLU inside the convolutional blocks.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class _NNConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o), nn.LeakyReLU(0.01, True),
            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o), nn.LeakyReLU(0.01, True),
        )
    def forward(self, x): return self.seq(x)

class nnUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=2):
        super().__init__()
        ch = [32, 64, 128, 256, 320]
        self.enc = nn.ModuleList([_NNConv(n_channels if i==0 else ch[i-1], ch[i]) for i in range(5)])
        self.pool = nn.MaxPool2d(2)
        self.ups  = nn.ModuleList([nn.ConvTranspose2d(ch[i], ch[i-1], 2, stride=2) for i in range(4, 0, -1)])
        self.dec  = nn.ModuleList([_NNConv(ch[i-1]*2, ch[i-1]) for i in range(4, 0, -1)])
        self.out  = nn.Conv2d(ch[0], n_classes, 1)

    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            x = enc(x)
            if i < 4: skips.append(x); x = self.pool(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            dy = skip.size(2) - x.size(2); dx = skip.size(3) - x.size(3)
            x = F.pad(x, [dx//2, dx - dx//2, dy//2, dy - dy//2])
            x = dec(torch.cat([skip, x], 1))
        return self.out(x)


### Model Initialization
Here is how to initialize nnU-Net:


In [ ]:
model = nnUNet(n_channels=3, n_classes=2)
print("nnU-Net instantiated successfully!")
